In [ ]:
# imports
import csv
from pathlib import Path
import pandas as pd
import numpy as np
import os
from scipy.stats import spearmanr, ttest_rel, pearsonr
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
import math
import pingouin as pg
import random
from datetime import datetime
from scipy.signal import resample
from scipy.interpolate import interp1d
import matplotlib as mpl

# Ensures consistent PDF output formatting (vector fonts)
mpl.rcParams["ps.useafm"] = True

In [ ]:
# Gets all day directories (like D01, D02, etc.) for a specific subject.
# Filters folder names that begin with "D", sorts them, and returns the list of paths.
def get_day_paths(subject_path: Path): # checked
    days = os.listdir(subject_path)
    day_paths = sorted(list(map(lambda x: subject_path / x, list(filter(lambda x: x[0] == "D", days)))))

    return day_paths

In [ ]:
# Reads the assigned.csv file inside the given day/segment folder 
# Returns a dictionary of DataFrames with keys: "affected finger", "affected wrist", "less affected finger", "less affected wrist
def read_assigned_sensor_csv(day_path: Path): # checked
    """
    Read all sensor data. Concatenate and order if there are more than one file.
    """
    assigned_data: pd.DataFrame = pd.read_csv(list(day_path.glob("assigned.csv"))[0])

    affected_finger: pd.DataFrame = assigned_data["affected finger"]
    affected_wrist: pd.DataFrame = assigned_data["affected wrist"]
    less_affected_finger: pd.DataFrame = assigned_data["less affected finger"]
    less_affected_wrist: pd.DataFrame = assigned_data["less affected wrist"]

    return {"affected finger": affected_finger,
            "affected wrist": affected_wrist,
            "less affected finger": less_affected_finger,
            "less affected wrist": less_affected_wrist}

In [ ]:
# Gets all subject directories inside the main data folder, ignoring hidden files
# For each subject folder:
    # Calls get_day_paths() to get all day paths.
    # Stores them in a dictionary where the key is the subject name and the value is a list of day paths.
def get_subject_day_paths(data_path: Path) -> dict[str, list[Path]]: # checked
    all_subject_dir_names: list[str] = os.listdir(data_path)
    all_subject_paths: list[Path] = sorted(list(
        map(lambda x: data_path / x, list(filter(lambda x: x[0] != ".", all_subject_dir_names)))
    ))
    subject_day_paths = {}

    for subject_path in all_subject_paths:
        subject = subject_path.name
        subject_day_paths[subject] = get_day_paths(subject_path=subject_path)

    return subject_day_paths

In [ ]:
# Prepares the output folder structure to store each subject/day combination.
def create_dirs(valid_subject_day_paths: dict[str, list[Path]], output_dir: Path): # checked
    subjects: list[str] = valid_subject_day_paths.keys()

    for subject in subjects:
        subject_output_path: Path = output_dir / subject
        if not os.path.exists(subject_output_path):
            os.makedirs(subject_output_path)

        day_paths: list[Path] = valid_subject_day_paths[subject]
        days: list[str] = list(map(lambda day_path: day_path.name, day_paths))

        for day in days:
            day_output_path: Path = output_dir / subject / day
            if not os.path.exists(day_output_path):
                os.makedirs(day_output_path)

In [ ]:
def band_pass_filter(input_np: np.ndarray):
    """
    Apply a 0.25–10 Hz band-pass Butterworth filter to remove noise and drift.

    Args:
        input_np (np.ndarray): Raw sensor signal array (e.g., accelerometer).

    Returns:
        np.ndarray: Filtered signal.
    """
    sample_rate = 50  # Hz (sampling rate)
    lowcut = 0.25
    highcut = 10
    nyq = 0.5 * sample_rate  # Nyquist frequency

    low = lowcut / nyq
    high = highcut / nyq

    # 6th order Butterworth band-pass filter
    b, a = butter(6, [low, high], btype="bandpass")
    bpf_np = filtfilt(b, a, input_np, axis=0)

    return bpf_np

In [ ]:
def absolute(input_np):
    """
    Take absolute value of signal (remove directionality).

    Args:
        input_np (np.ndarray): Filtered signal.

    Returns:
        np.ndarray: Absolute values of input.
    """
    return np.abs(input_np)

In [ ]:
def average_epoch_50(input_np):
    """
    Compute moving average over 50-sample windows (1 second if 50 Hz).

    Args:
        input_np (np.ndarray): Absolute signal array.

    Returns:
        np.ndarray: Downsampled averaged signal.
    """
    average_epoch_50_np = np.cumsum(input_np, axis=-1, dtype=float)
    average_epoch_50_np[50:] = average_epoch_50_np[50:] - average_epoch_50_np[:-50]
    average_epoch_50_np = average_epoch_50_np[50 - 1 :: 50] / 50

    return average_epoch_50_np

In [ ]:
def process_data_pipeline(raw):
    """
    Full sensor processing pipeline (filters + transforms).

    Steps:
      1. Band-pass filter (0.25–10 Hz)
      2. Take absolute value
      3. Compute average in 50-sample windows

    Args:
        raw (np.ndarray): Raw accelerometer signal.

    Returns:
        np.ndarray: Processed and downsampled signal.
    """
    bpf_np = band_pass_filter(raw)
    absolute_np = absolute(bpf_np)
    average_epoch_50_np = average_epoch_50(absolute_np)
    return average_epoch_50_np

In [ ]:
def process_data(assigned_dir: Path, output_dir: Path, pipeline: callable):
    """
    Apply a given processing pipeline to all assigned data.

    For each subject/day:
        - Reads 'assigned.csv'
        - Processes each sensor column using the provided pipeline
        - Saves the result as 'processed.csv'

    Args:
        assigned_dir (Path): Path containing 'assigned.csv' files.
        output_dir (Path): Directory to save processed files.
        pipeline (callable): Function applied to sensor data (e.g., process_data_pipeline).
    """
    valid_subject_day_paths: dict[str, list[Path]] = get_subject_day_paths(data_path=assigned_dir)
    create_dirs(valid_subject_day_paths=valid_subject_day_paths, output_dir=output_dir)

    for subject, day_paths in valid_subject_day_paths.items():
        for day_path in day_paths:
            print(subject, day_path.name)
            assigned_day_dict: dict[str, pd.DataFrame] = read_assigned_sensor_csv(day_path)

            # Apply pipeline to each data stream
            processed_day_dict: dict[str, pd.DataFrame] = {
                "affected finger": pipeline(assigned_day_dict["affected finger"]),
                "affected wrist": pipeline(assigned_day_dict["affected wrist"]),
                "less affected finger": pipeline(assigned_day_dict["less affected finger"]),
                "less affected wrist": pipeline(assigned_day_dict["less affected wrist"])
            }

            # Combine processed columns into one DataFrame
            df_processed_day = pd.DataFrame({
                "affected finger": pd.Series(processed_day_dict["affected finger"]),
                "affected wrist": pd.Series(processed_day_dict["affected wrist"]),
                "less affected finger": pd.Series(processed_day_dict["less affected finger"]),
                "less affected wrist": pd.Series(processed_day_dict["less affected wrist"])
            })

            # Save processed data for this day
            df_processed_day.to_csv(output_dir / subject / day_path.name / "processed.csv", index=False)

In [ ]:
process_data(assigned_dir=Path("path/to/your/data"), output_dir=Path("path/to/your/data"), pipeline=process_data_pipeline)